# Eval Harness
# 0. 介绍

**研究背景**：Agent 完成真实任务时，不只会生成一段回答，还会调用工具并改变文件、数据库、网页或其他环境状态。外层程序必须把任务目标、执行过程和最终环境证据放在一起检查，才能回答两个基本问题：任务是否真的完成，以及完成过程是否可靠、合规且可复现。

**现存问题**：生产中常见的错误基线是只检查模型是否说了“完成”、是否发出工具调用、接口是否返回成功，或者再让另一个大模型给最终文字打分。这样会把“模型想做什么”误当成“环境里实际发生了什么”：代码 Agent 可能声称已经修复但目标测试没有通过，业务 Agent 可能声称已经提交但数据库状态没有改变。单次运行和单一总分还会掩盖越权步骤、无效重试、环境噪声与评分器自身的漏判或抖动，最终产生无法诊断的虚假成功或虚假失败。

**解决方案**：本 Notebook 将实现一个极简的 Eval Harness，采用`明确任务规范 + 受控重复 Rollout + 多层 Grader + Trace 原生评估`机制：先固定初始状态、允许动作、预算和可执行的成功标准；再用真实 API 重复运行并保存完整轨迹；由 Outcome Grader 直接检查测试、文件或环境终态，由 Trajectory Grader 检查工具链、权限和预算，由 Evaluator Grader 用已知正确与错误样本验证评分器本身；最后报告成功率、方差、Token、成本、延迟和停止原因，并把生产 Trace 转为回归用例。该方案与 SWE-bench、Terminal-Bench、WebArena、OSWorld 的环境结果验证，以及 LangChain Deep-Agent Evaluation、Anthropic 2026 eval 工程实践的分层评估方向一致；语义难以确定时才把校准后的 LLM-as-Judge 作为补充，而不把模型自评当作唯一证据。后文将用同一份真实 API 决策对比：基线版本只检查模型意图而误报成功，改进版本核验真实产物并给出可追溯判定，从而直观看到可靠评估的关键不是“再问模型一次”，而是“用独立证据证明任务确实完成”。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以调用的工具、需要完成的任务，以及判断工具结果是否正确的标准。

# 2. 前置准备
## 2.1 写出可测任务
评估必须先规定什么叫成功，否则同一个结果可以被不同的人解释成不同分数。本节准备一个最小文件修复任务：模型只负责提交完整的新文件，后面的 Harness 再根据文件运行结果判断任务是否完成。

In [2]:
import json

# initial_state 是每次评估共同使用的错误起点。
# success_criteria 描述环境最后必须留下的客观结果。
BROKEN_SOURCE = '''def submit_application(form):
    return {"status": "draft", "confirmation_id": None}
'''
TARGET_SOURCE = '''def submit_application(form):
    return {"status": "submitted", "confirmation_id": "EV-2026-0803"}
'''
SUCCESS_CRITERIA = {"status": "submitted", "confirmation_id": "EV-2026-0803"}
TASK = {
    "task_id": "eval_file_fix_001",
    "instruction": "修复 portal.py，使提交申请后状态为 submitted，并生成确认编号 EV-2026-0803。",
    "initial_state": {"portal.py": BROKEN_SOURCE},
    "allowed_tools": ["repair_file"],
    "success_criteria": SUCCESS_CRITERIA,
    "budget": {"max_steps": 2, "context_tokens": 1200},
}
print(json.dumps({"task_id": TASK["task_id"],
                  "instruction": TASK["instruction"],
                  "initial_files": list(TASK["initial_state"]),
                  "success_criteria": TASK["success_criteria"],
                  "budget": TASK["budget"]}, ensure_ascii=False, indent=2))

{
  "task_id": "eval_file_fix_001",
  "instruction": "修复 portal.py，使提交申请后状态为 submitted，并生成确认编号 EV-2026-0803。",
  "initial_files": [
    "portal.py"
  ],
  "success_criteria": {
    "status": "submitted",
    "confirmation_id": "EV-2026-0803"
  },
  "budget": {
    "max_steps": 2,
    "context_tokens": 1200
  }
}


输出把评估刻度固定下来：起点只有一个错误文件，允许的动作是提交修复文件，成功必须同时满足状态和确认编号。下一步把这个动作写成模型能理解的工具协议。

## 2.2 声明唯一修复工具
模型不能直接修改工作区，它只能返回结构化的工具请求。本节声明 `repair_file`，要求模型给出文件路径和完整文件内容；真正写入文件会在后面的受控执行阶段发生。

In [3]:
# path 限定本任务唯一允许修改的文件。
# content 让执行器可以完整重建文件，而不是猜测模型想改哪一行。
tools = [{
    "type": "function",
    "function": {
        "name": "repair_file",
        "description": "提交修复后的完整 Python 文件",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "enum": ["portal.py"]},
                "content": {"type": "string"},
            },
            "required": ["path", "content"],
        },
    },
}]
print({"tool": tools[0]["function"]["name"],
       "required": tools[0]["function"]["parameters"]["required"]})

{'tool': 'repair_file', 'required': ['path', 'content']}


输出说明模型必须提交两个字段：要改哪个文件，以及改完后的完整内容。工具协议只负责表达动作，不负责证明动作已经生效；下一步把任务和初始文件放进输入上下文。

## 2.3 组装模型输入上下文
评估时所有 rollout 必须看到同一份任务和同一份初始文件，否则比较的就不是同一个问题。本节固定 system 指令、用户任务和上下文预算，下一章会把它们发送给真实 API。

In [4]:
# system 说明模型必须提交工具请求，不能把自我描述当作任务结果。
# user 保留任务目标和当前源码，保证输入上下文可以被完整重放。
context_budget = TASK["budget"]["context_tokens"]
messages = [
    {
        "role": "system",
        "content": "你是代码修复助手。必须调用 repair_file 提交修复后的完整文件，不要声称测试已经通过。",
    },
    {
        "role": "user",
        "content": TASK["instruction"] + "\n当前 portal.py：\n" + BROKEN_SOURCE,
    },
]
context_chars = sum(len(message["content"]) for message in messages)
print({"roles": [message["role"] for message in messages],
       "context_chars": context_chars,
       "token_estimate": context_chars // 4,
       "token_budget": context_budget,
       "tool": tools[0]["function"]["name"]})

{'roles': ['system', 'user'], 'context_chars': 203, 'token_estimate': 50, 'token_budget': 1200, 'tool': 'repair_file'}


输出显示输入由 system 和 user 两条消息组成，任务、错误源码和工具名称都已经进入上下文，估算 token 也低于预算。至此，任务定义、动作协议和输入上下文全部固定；下一章将请求真实模型并记录它的结构化决定。

# 3. 获取并验证 API 响应
## 3.1 发送真实请求
任务和工具说明已经固定。本节把同一份输入交给 `.env` 中配置的真实模型，并要求模型必须返回一个 `repair_file` 工具请求；这里只记录模型决定，不修改工作区。

In [5]:
from time import perf_counter

# perf_counter 记录从发出请求到收到响应的真实等待时间。
# tool_choice=require 让模型必须表达一个结构化动作，而不是只返回口头回答。
request_started = perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
    tool_choice="required",
    temperature=0,
    max_tokens=320,
)
api_latency_ms = round((perf_counter() - request_started) * 1000)
print("真实 API 响应已收到")

真实 API 响应已收到


输出表示真实模型已经返回，但“收到响应”只说明网络调用完成，并不等于文件已经修好。下一步把响应中的工具名称和参数还原成普通 Python 字典，后续执行器才能使用同一份决定。

## 3.2 读取结构化模型决定
工具调用的参数是 JSON 字符串，不能直接当作 Python 数据使用。本节只做一次解析，并保留调用 ID、工具名称和参数，确保之后的执行结果可以和这次模型决定对应起来。

In [6]:
# choice 保存本次响应的第一条模型选择。
# call_id 把模型请求与后面的工具结果连接成同一条执行记录。
choice = response.choices[0]
tool_call = choice.message.tool_calls[0]
model_decision = {
    "call_id": tool_call.id,
    "tool": tool_call.function.name,
    "arguments": json.loads(tool_call.function.arguments),
}
print(json.dumps(model_decision, ensure_ascii=False, indent=2))

{
  "call_id": "call_f3bc54186c9a43cb89f7ee0e",
  "tool": "repair_file",
  "arguments": {
    "path": "portal.py",
    "content": "def submit_application(form):\n    return {\"status\": \"submitted\", \"confirmation_id\": \"EV-2026-0803\"}"
  }
}


输出把模型的意图变成了可观察的三部分：调用编号、工具名称和文件参数。此时只能说模型提交了修复申请，不能说环境已经产生成功产物；下一步补齐这次真实请求的运行指标。

## 3.3 保存本次运行指标
评估报告不能只保存模型文字，还要知道这次调用用了哪个 provider、多少 Token 和多长时间。本节直接读取 API 返回的 usage，并把成本未知明确记录为未知，不用估算数字填空。

In [7]:
# usage 使用 provider 返回的真实 Token 统计，而不是按字符猜测。
# stop_reason 记录模型为何停止，不能把它解释成任务成功。
usage = response.usage
api_metrics = {
    "provider": config["NANO_BACKEND"],
    "model": model_name,
    "input_tokens": usage.prompt_tokens,
    "output_tokens": usage.completion_tokens,
    "total_tokens": usage.total_tokens,
    "cost_usd": None,
    "latency_ms": api_latency_ms,
    "stop_reason": choice.finish_reason,
}
print(json.dumps(api_metrics, ensure_ascii=False, indent=2))

{
  "provider": "openai",
  "model": "LongCat-2.0",
  "input_tokens": 234,
  "output_tokens": 156,
  "total_tokens": 390,
  "cost_usd": null,
  "latency_ms": 4673,
  "stop_reason": "tool_calls"
}


输出记录了本次真实 API 调用的来源、Token、延迟、停止原因和未知成本。现在模型决定与运行指标都已保存，但 `portal.py` 仍保持错误初始状态；下一章将展示只看模型意图的基线为什么会误报成功。

# 4. 定义基线组件 *
## 4.1 只检查模型是否提出动作
生产中一种常见的错误评估方式是：只要模型返回了预期工具名称，就把任务记为成功。这个做法实现简单，但它检查的是模型的行动意图，不是工具是否执行，也不是环境是否真的改变。下面把这条错误规则写成最小基线评估器。

In [8]:
def grade_intent_only(decision):
    # 基线只比较工具名称，不执行模型提交的文件修改。
    # 因此只要模型提出预期动作，基线就直接返回成功。
    proposed_expected_tool = decision["tool"] == TASK["allowed_tools"][0]
    return {
        "success": proposed_expected_tool,
        "reason": "模型提出了预期修复动作",
    }

print("基线评估器已定义：只检查模型意图")

基线评估器已定义：只检查模型意图


输出说明基线评估器已经准备好，但还没有给本次运行打分。它的输入只有模型决定，完全看不到 `portal.py` 的真实状态；下一章将运行它，并把报告分数与环境产物并排比较。

# 5. 展示基线故障 *
## 5.1 让基线给真实模型决定打分
第 3 章的真实模型已经提出 `repair_file` 动作。现在把这份决定交给第 4 章的基线评估器；由于基线只看工具名称，它会立刻给出成功结论。

In [9]:
# 输入是本次真实 API 返回的结构化模型决定。
# 基线只评分，不执行 repair_file，也不会改变初始程序。
baseline_report = grade_intent_only(model_decision)
print(json.dumps(baseline_report, ensure_ascii=False, indent=2))

{
  "success": true,
  "reason": "模型提出了预期修复动作"
}


输出中的 `success` 为 `true`，理由只是模型提出了预期动作。这个分数没有引用任何执行结果；下一步直接运行当前初始程序，查看环境真正返回了什么。

## 5.2 读取真实环境产物
模型的工具请求尚未执行，因此环境里仍然是第 2 章定义的错误源码。本节直接加载这份源码并调用 `submit_application`，得到与模型自述无关的实际结果。

In [10]:
# namespace 保存当前源码运行后定义的函数。
# actual_artifact 来自程序真实返回值，不读取模型的成功声明。
namespace = {}
exec(BROKEN_SOURCE, namespace)
actual_artifact = namespace["submit_application"]({})
print(json.dumps(actual_artifact, ensure_ascii=False, indent=2))

{
  "status": "draft",
  "confirmation_id": null
}


环境返回的状态仍是 `draft`，确认编号仍为空，说明任务没有完成。基线报告和环境事实已经互相冲突；下一步把冲突整理成一条明确的假阳性记录。

## 5.3 对照报告分数与环境事实
当评估器报告成功，但实际产物不符合第 2 章的成功标准时，这次评分就是假阳性。本节只比较这两个已经得到的结果，不改变任何数据。

In [11]:
# artifact_passed 表示环境结果是否达到统一成功标准。
# false_positive 表示评估器报成功，但真实任务没有成功。
artifact_passed = actual_artifact == SUCCESS_CRITERIA
baseline_comparison = {
    "reported_success": baseline_report["success"],
    "artifact_passed": artifact_passed,
    "false_positive": baseline_report["success"] and not artifact_passed,
}
print(json.dumps(baseline_comparison, ensure_ascii=False, indent=2))

{
  "reported_success": true,
  "artifact_passed": false,
  "false_positive": true
}


输出显示 `reported_success=true`、`artifact_passed=false`、`false_positive=true`。模型正确提出了修复动作，但错误基线跳过执行和环境检查，因而把“准备修复”误报为“已经修好”。下一章将定义以真实环境产物为准的改进组件。

# 6. 定义改进组件 *
## 6.1 把模型动作执行到环境
改进评估的第一步不是再次询问模型，而是实际执行它提出的动作。下面的执行器从同一个错误初始状态开始，把模型提交的完整源码写入任务工作区，并记录文件是否真的改变。

In [12]:
def execute_repair(decision, initial_state):
    # 使用副本保存本次 rollout 的状态，不改写任务定义中的初始状态。
    # tool_result 记录调用 ID 和状态变化，供后面的 trace 直接复用。
    workspace = initial_state.copy()
    path = decision["arguments"]["path"]
    old_content = workspace[path]
    new_content = decision["arguments"]["content"]
    workspace[path] = new_content
    tool_result = {
        "call_id": decision["call_id"],
        "path": path,
        "state_changed": old_content != new_content,
    }
    return workspace, tool_result

print("受控执行器已定义")

受控执行器已定义


输出说明执行器已经定义，但它还没有修改任何工作区。执行后是否发生状态变化只能证明文件被改过，仍不能证明任务正确；下一步定义直接检查程序运行结果的 Outcome Grader。

## 6.2 用环境产物判断成功
当前学术 benchmark 和生产评估的核心共识，是优先使用可执行测试、文件内容或环境终态等独立证据。下面的 Outcome Grader 会运行工作区中的 `portal.py`，再把真实返回值与第 2 章固定的成功标准比较。

In [13]:
def grade_outcome(workspace):
    # 运行工作区中的源码，得到环境实际提供的函数。
    # success 只由真实产物与统一成功标准的比较结果决定。
    namespace = {}
    exec(workspace["portal.py"], namespace)
    artifact = namespace["submit_application"]({})
    return {
        "success": artifact == SUCCESS_CRITERIA,
        "artifact": artifact,
    }

print("Outcome Grader 已定义")

Outcome Grader 已定义


输出说明产物评分器已经准备好。它不读取模型的成功声明，只读取修复后程序真正返回的状态和确认编号；下一步定义最小 trace，把模型、工具和评分证据连成一条可追溯记录。

## 6.3 连接模型、工具与评分证据
只保存最终分数仍然无法解释成功或失败发生在哪一步。下面构造三条最小 trace：模型提出动作、工具改变状态、Grader 检查产物；相同的调用 ID 会把前两步关联起来。

In [14]:
def build_eval_trace(decision, tool_result, outcome):
    # model 与 tool 使用同一个 call_id，表示它们属于同一次动作。
    # grader 保存实际产物和分数，使最终结论可以回到环境证据。
    return [
        {"step": 1, "type": "model", "call_id": decision["call_id"],
         "tool": decision["tool"]},
        {"step": 2, "type": "tool", "call_id": tool_result["call_id"],
         "path": tool_result["path"], "state_changed": tool_result["state_changed"]},
        {"step": 3, "type": "grader", "success": outcome["success"],
         "artifact": outcome["artifact"]},
    ]

print("评估 Trace Builder 已定义")

评估 Trace Builder 已定义


输出说明改进组件已经齐备：执行器负责产生状态变化，Outcome Grader 负责依据环境产物打分，Trace Builder 负责保留因果链。下一章将用第 3 章的真实模型决定运行这三部分，并展示修复后的结果。

# 7. 展示修复结果 *
## 7.1 执行真实模型提出的修复
现在把第 3 章真实模型返回的 `repair_file` 决定交给受控执行器。执行器从固定错误状态开始，将模型提供的完整源码写入工作区，并返回本次动作是否真的改变了文件。

In [15]:
# 每次都从 TASK.initial_state 开始，保证与基线使用同一个起点。
# tool_result 保存调用关联和文件状态变化，不包含成功自评。
fixed_workspace, tool_result = execute_repair(model_decision, TASK["initial_state"])
print(json.dumps(tool_result, ensure_ascii=False, indent=2))

{
  "call_id": "call_f3bc54186c9a43cb89f7ee0e",
  "path": "portal.py",
  "state_changed": true
}


输出中的路径是 `portal.py`，`state_changed=true` 表示模型动作已经真正改变工作区。文件改变仍不等于任务成功；下一步运行修复后的程序，并依据环境产物打分。

## 7.2 检查修复后的真实产物
Outcome Grader 不读取工具名称或模型声明，而是直接执行工作区中的新程序。只有真实返回值同时满足状态和确认编号，改进版本才会报告成功。

In [16]:
# fixed_workspace 包含真实模型提交并由执行器写入的新源码。
# fixed_outcome 同时保存客观产物和由该产物得到的分数。
fixed_outcome = grade_outcome(fixed_workspace)
print(json.dumps(fixed_outcome, ensure_ascii=False, indent=2))

{
  "success": true,
  "artifact": {
    "status": "submitted",
    "confirmation_id": "EV-2026-0803"
  }
}


输出显示 `success=true`，环境实际返回 `submitted` 和正确确认编号。这个成功结论来自修复后程序的运行结果，而不是模型说自己完成了任务；下一步把整条因果链整理成 trace。

## 7.3 展示完整评估 Trace
最终分数需要能够追溯。下面把模型决定、工具状态变化和 Grader 产物按发生顺序排成三条记录；前两条共享同一个调用 ID，最后一条保存独立环境证据。

In [17]:
# fixed_trace 按 model、tool、grader 的顺序保留三类证据。
# 多行循环逐条打印，便于直接观察每一步的数据流转。
fixed_trace = build_eval_trace(model_decision, tool_result, fixed_outcome)
for record in fixed_trace:
    print(json.dumps(record, ensure_ascii=False))

{"step": 1, "type": "model", "call_id": "call_f3bc54186c9a43cb89f7ee0e", "tool": "repair_file"}
{"step": 2, "type": "tool", "call_id": "call_f3bc54186c9a43cb89f7ee0e", "path": "portal.py", "state_changed": true}
{"step": 3, "type": "grader", "success": true, "artifact": {"status": "submitted", "confirmation_id": "EV-2026-0803"}}


三条输出重建了完整路径：真实模型提出修复，执行器确实改变文件，Grader 从环境中读到正确产物并判定成功。与第 5 章相比，模型没有变化，变化的是外层 Harness 补上了执行和环境结果验证；下一章将汇总两条路径的消融差异。

# 8. 汇总消融对照
## 8.1 检查执行轨迹
Outcome Grader 回答任务是否完成，Trajectory Grader 则回答完成路径是否完整。下面检查 trace 的步骤顺序、调用关联、状态变化和步数预算，避免只看最后一个成功字段。

In [18]:
# trace_types 保留真实执行步骤的先后顺序。
# trajectory_grade 分别检查关联、状态变化和任务步数预算。
trace_types = []
for record in fixed_trace:
    trace_types.append(record["type"])

trajectory_grade = {
    "ordered_steps": trace_types == ["model", "tool", "grader"],
    "call_correlated": fixed_trace[0]["call_id"] == fixed_trace[1]["call_id"],
    "state_changed": fixed_trace[1]["state_changed"],
    "within_budget": len(fixed_trace) - 1 <= TASK["budget"]["max_steps"],
}
trajectory_grade["success"] = all(trajectory_grade.values())
print(json.dumps(trajectory_grade, ensure_ascii=False, indent=2))

{
  "ordered_steps": true,
  "call_correlated": true,
  "state_changed": true,
  "within_budget": true,
  "success": true
}


输出中的四项检查均为 `true`，说明模型动作、工具结果和 Grader 证据顺序完整，并且没有超出两步动作预算。轨迹可靠之后，还要确认负责打分的 Outcome Grader 本身没有把明显错误样本判成成功。

## 8.2 检查评分器本身
评分器也可能成为误差来源。最小 Evaluator Grader 使用一份已知错误源码和一份已知正确源码作为探针：前者必须失败，后者必须通过，才说明这把尺子能区分当前任务的成败。

In [19]:
# known_bad_grade 检查评分器能否拒绝原始错误程序。
# known_good_grade 检查评分器能否接受满足标准的正确程序。
known_bad_grade = grade_outcome(TASK["initial_state"])
known_good_grade = grade_outcome({"portal.py": TARGET_SOURCE})
evaluator_grade = {
    "known_bad_rejected": not known_bad_grade["success"],
    "known_good_accepted": known_good_grade["success"],
}
evaluator_grade["success"] = all(evaluator_grade.values())
print(json.dumps(evaluator_grade, ensure_ascii=False, indent=2))

{
  "known_bad_rejected": true,
  "known_good_accepted": true,
  "success": true
}


输出显示错误样本被拒绝、正确样本被接受，Evaluator Grader 因此通过。这个结果只证明当前确定性成功标准可以被稳定区分；下一步用真实 API 重复运行同一任务，观察模型与基础设施带来的波动。

## 8.3 重复真实 Rollout
单次成功不能代表稳定成功。为了复用与第 3 章完全相同的请求条件，下面先把一次真实模型请求封装成函数；函数返回结构化决定和本次 API 指标，不执行工具也不评分。

In [20]:
import openai

def request_repair_decision():
    # 每次请求复用同一个模型、消息、工具和采样参数。
    # 返回值同时保存模型动作与 provider 提供的运行指标。
    started = perf_counter()
    current_response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        tools=tools,
        tool_choice="required",
        temperature=0,
        max_tokens=320,
    )
    latency_ms = round((perf_counter() - started) * 1000)
    current_choice = current_response.choices[0]
    current_call = current_choice.message.tool_calls[0]
    current_usage = current_response.usage
    decision = {
        "call_id": current_call.id,
        "tool": current_call.function.name,
        "arguments": json.loads(current_call.function.arguments),
    }
    metrics = {
        "provider": config["NANO_BACKEND"],
        "model": model_name,
        "input_tokens": current_usage.prompt_tokens,
        "output_tokens": current_usage.completion_tokens,
        "total_tokens": current_usage.total_tokens,
        "cost_usd": None,
        "latency_ms": latency_ms,
        "stop_reason": current_choice.finish_reason,
    }
    return decision, metrics

print("重复 Rollout 请求函数已定义")

重复 Rollout 请求函数已定义


输出说明重复请求函数已经定义，但没有新增 API 调用。下一格把第 3 章的真实 rollout 作为第一次样本，再发起两次真实请求，并让每次决定都经过相同的执行器、Grader 和 Trace Builder。

### 8.3.1 连续运行三次
三次 rollout 使用相同任务、初始状态、模型参数和成功标准。每次运行都会独立请求真实模型、执行文件修改、检查环境产物并保存 trace；下面逐次打印成功状态、产物、Token 和延迟。

In [21]:
# 第一次样本直接复用第 3 章本次真实 API 响应和第 7 章结果。
# 后两次样本重新请求真实模型，并走完完全相同的执行与评分流程。
rollouts = [{
    "run_id": "run-1",
    "decision": model_decision,
    "metrics": api_metrics,
    "outcome": fixed_outcome,
    "trace": fixed_trace,
}]

for run_number in range(2, 4):
    repeated_decision, repeated_metrics = request_repair_decision()
    repeated_workspace, repeated_tool = execute_repair(repeated_decision, TASK["initial_state"])
    repeated_outcome = grade_outcome(repeated_workspace)
    repeated_trace = build_eval_trace(repeated_decision, repeated_tool, repeated_outcome)
    rollouts.append({
        "run_id": f"run-{run_number}",
        "decision": repeated_decision,
        "metrics": repeated_metrics,
        "outcome": repeated_outcome,
        "trace": repeated_trace,
    })

for rollout in rollouts:
    print(json.dumps({
        "run_id": rollout["run_id"],
        "success": rollout["outcome"]["success"],
        "artifact": rollout["outcome"]["artifact"],
        "total_tokens": rollout["metrics"]["total_tokens"],
        "latency_ms": rollout["metrics"]["latency_ms"],
        "stop_reason": rollout["metrics"]["stop_reason"],
    }, ensure_ascii=False))

{"run_id": "run-1", "success": true, "artifact": {"status": "submitted", "confirmation_id": "EV-2026-0803"}, "total_tokens": 390, "latency_ms": 4673, "stop_reason": "tool_calls"}
{"run_id": "run-2", "success": true, "artifact": {"status": "submitted", "confirmation_id": "EV-2026-0803"}, "total_tokens": 390, "latency_ms": 4191, "stop_reason": "tool_calls"}
{"run_id": "run-3", "success": true, "artifact": {"status": "submitted", "confirmation_id": "EV-2026-0803"}, "total_tokens": 387, "latency_ms": 4467, "stop_reason": "tool_calls"}


三条输出都来自真实 API rollout，并且都留下相同的正确环境产物。Token 和延迟仍可能不同，这正是重复运行要暴露的波动；下一步把成功率、成功方差和延迟方差汇总成数字。

## 8.4 汇总成功率与方差
只列出三次明细不便比较。本节把成功转换为 0/1，计算成功率和总体方差，同时汇总延迟均值、延迟方差和总 Token；三次样本只用于展示机制，不代表生产 SLA。

In [22]:
from statistics import mean, pvariance

# success_values 用 1 和 0 表示每次 rollout 是否通过。
# latency_values 与 total_tokens 来自每次真实 API 响应。
success_values = []
latency_values = []
total_tokens = 0
for rollout in rollouts:
    success_values.append(int(rollout["outcome"]["success"]))
    latency_values.append(rollout["metrics"]["latency_ms"])
    total_tokens += rollout["metrics"]["total_tokens"]

variance_report = {
    "runs": len(rollouts),
    "success_rate": round(mean(success_values), 3),
    "success_variance": round(pvariance(success_values), 3),
    "latency_mean_ms": round(mean(latency_values)),
    "latency_variance_ms2": round(pvariance(latency_values)),
    "total_tokens": total_tokens,
    "cost_usd": None,
}
print(json.dumps(variance_report, ensure_ascii=False, indent=2))

{
  "runs": 3,
  "success_rate": 1,
  "success_variance": 0,
  "latency_mean_ms": 4444,
  "latency_variance_ms2": 38993,
  "total_tokens": 1167,
  "cost_usd": null
}


输出给出了三次真实运行的成功率和方差。成功方差为零表示这组三次任务结果一致；延迟方差保留了基础设施波动，成本因 provider 未返回金额而明确记为 `null`。下一步回到同一次模型决定，比较有无环境验证时的结论差异。

## 8.5 汇总基线与改进版本
消融必须固定模型决定，才能把差异归因给 Harness。下面让基线和改进版本共享第 3 章同一次真实 API 调用及其 Token、延迟和成本，只改变动作是否执行、成功是否依据环境产物。

In [23]:
# 两条路径使用相同 provider、model 和真实 API 决定。
# artifact_passed 与 false_positive 直接显示 Harness 机制带来的差异。
ablation_rows = [
    {
        "version": "基线：只看模型意图",
        "provider": api_metrics["provider"],
        "model": api_metrics["model"],
        "total_tokens": api_metrics["total_tokens"],
        "latency_ms": api_metrics["latency_ms"],
        "cost_usd": api_metrics["cost_usd"],
        "reported_success": baseline_report["success"],
        "artifact_passed": artifact_passed,
        "false_positive": baseline_comparison["false_positive"],
    },
    {
        "version": "改进：执行并检查环境产物",
        "provider": api_metrics["provider"],
        "model": api_metrics["model"],
        "total_tokens": api_metrics["total_tokens"],
        "latency_ms": api_metrics["latency_ms"],
        "cost_usd": api_metrics["cost_usd"],
        "reported_success": fixed_outcome["success"],
        "artifact_passed": fixed_outcome["success"],
        "false_positive": False,
    },
]
for row in ablation_rows:
    print(json.dumps(row, ensure_ascii=False))

{"version": "基线：只看模型意图", "provider": "openai", "model": "LongCat-2.0", "total_tokens": 390, "latency_ms": 4673, "cost_usd": null, "reported_success": true, "artifact_passed": false, "false_positive": true}
{"version": "改进：执行并检查环境产物", "provider": "openai", "model": "LongCat-2.0", "total_tokens": 390, "latency_ms": 4673, "cost_usd": null, "reported_success": true, "artifact_passed": true, "false_positive": false}


两条路径使用完全相同的模型调用，因此 Token、延迟和成本相同；基线却产生假阳性，改进版则用真实产物证明成功。差异来自 Harness 是否执行动作并核验环境，而不是更换了更强模型。接下来保存全部 trace 和 eval report。

## 8.6 保存 Trace JSONL
Trace 是评估输入，而不是附属日志。本节把三次真实 rollout 的九条 model、tool、grader 记录写入项目 `traces/` 目录；每条记录附带 run ID、provider、model 和本次 API 指标。

In [24]:
from pathlib import Path

# trace_lines 按 rollout 和步骤顺序保存九条结构化证据。
# 公开产物只记录运行信息，不保存 API key 或 base URL。
project_root = Path(find_dotenv()).parent
trace_path = project_root / "traces" / "06_V_nanoEvalHarness_minimal_trace.jsonl"
trace_lines = []
for rollout in rollouts:
    for record in rollout["trace"]:
        saved_record = record.copy()
        saved_record["run_id"] = rollout["run_id"]
        saved_record["provider"] = rollout["metrics"]["provider"]
        saved_record["model"] = rollout["metrics"]["model"]
        saved_record["total_tokens"] = rollout["metrics"]["total_tokens"]
        saved_record["latency_ms"] = rollout["metrics"]["latency_ms"]
        saved_record["cost_usd"] = rollout["metrics"]["cost_usd"]
        saved_record["stop_reason"] = rollout["metrics"]["stop_reason"]
        trace_lines.append(json.dumps(saved_record, ensure_ascii=False, sort_keys=True))
trace_path.write_text("\n".join(trace_lines) + "\n", encoding="utf-8")
print({"path": str(trace_path.relative_to(project_root)), "records": len(trace_lines)})

{'path': 'traces/06_V_nanoEvalHarness_minimal_trace.jsonl', 'records': 9}


输出显示 JSONL 文件路径和九条记录，正好对应三次 rollout 各三个步骤。下一步把任务、运行指标、多层 Grader、方差和消融结果汇总到一个 eval report，便于从单个文件读取最终结论。

## 8.7 保存 Eval Report
Eval report 是本次实验的结构化总结。下面保存 provider、model、依赖版本、三次真实运行、Grader 结果、方差、消融对照、trace 路径和最终成功产物；未知成本继续保留为 `null`。

In [25]:
# run_summaries 保留每次真实 rollout 的结果与 API 指标。
# eval_report 汇总任务、Grader、方差、消融和持久化 trace。
run_summaries = []
for rollout in rollouts:
    run_summaries.append({
        "run_id": rollout["run_id"],
        "success": rollout["outcome"]["success"],
        "artifact": rollout["outcome"]["artifact"],
        "input_tokens": rollout["metrics"]["input_tokens"],
        "output_tokens": rollout["metrics"]["output_tokens"],
        "total_tokens": rollout["metrics"]["total_tokens"],
        "latency_ms": rollout["metrics"]["latency_ms"],
        "cost_usd": rollout["metrics"]["cost_usd"],
        "stop_reason": rollout["metrics"]["stop_reason"],
    })

eval_report = {
    "notebook": "06_V_nanoEvalHarness_minimal.ipynb",
    "layer": "V",
    "task_id": TASK["task_id"],
    "provider": config["NANO_BACKEND"],
    "model": model_name,
    "openai_version": openai.__version__,
    "real_api_rollouts": len(rollouts),
    "runs": run_summaries,
    "trajectory_grade": trajectory_grade,
    "evaluator_grade": evaluator_grade,
    "variance": variance_report,
    "ablation": ablation_rows,
    "trace_file": str(trace_path.relative_to(project_root)),
    "successful_artifact": fixed_outcome["artifact"],
}
eval_path = project_root / "evals" / "06_V_nanoEvalHarness_minimal_eval.json"
eval_path.write_text(json.dumps(eval_report, ensure_ascii=False, indent=2), encoding="utf-8")
print({"path": str(eval_path.relative_to(project_root)),
       "real_api_rollouts": eval_report["real_api_rollouts"],
       "success_rate": eval_report["variance"]["success_rate"],
       "artifact": eval_report["successful_artifact"]})

{'path': 'evals/06_V_nanoEvalHarness_minimal_eval.json', 'real_api_rollouts': 3, 'success_rate': 1, 'artifact': {'status': 'submitted', 'confirmation_id': 'EV-2026-0803'}}


输出给出 eval report 路径、三次真实 rollout、成功率和最终产物。至此，错误基线的假阳性、改进版的环境验证、多层 Grader、重复运行、方差、trace 与结构化报告已经形成一个可从头运行的最小闭环。

## 8.8 拓展

### nano 版省略了什么

nano 版只有少量 rollout、确定性 grader 和简单方差，没有分层任务集、盲评、统计功效、置信区间、污染检查、人类校准、对抗样本和跨模型比较。生产 Eval Harness 必须同时评估 outcome、trajectory、资源与安全，并冻结任务、环境、grader 和 Harness 版本以支持可重复结论。

### 延伸阅读


1. 2026, [Anthropic, Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents)：Agent eval 的任务、trial、grader、transcript 与评测设计。
2. 2026, [On Randomness in Agentic Evals](https://arxiv.org/abs/2602.07150)：Agent 评测随机性、重复运行与统计报告。
3. 2024, [AgentDojo](https://arxiv.org/abs/2406.13352)：动态任务环境中同时评估效用和安全性。